In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"D:\AI Eng\Projects\RNN-Sentiment Analysis\data\IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

### Pre-processing

#### Removing URLs

In [8]:
import re # Regular Expressions
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text
df["review"]=df["review"].apply(remove_urls)

#### Removing Punctuations

In [9]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
df["review"]=df["review"].apply(remove_punctuation)

#### Removing HTML

In [10]:
def removing_html(text):
    text=re.sub(r"<.>","",text)
    return text
df["review"]=df["review"].apply(removing_html)

In [11]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production br br The filmin...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically theres a family where a little boy J...,negative
4,Petter Matteis Love in the Time of Money is a ...,positive


#### Removing the Stopswords

In [12]:
!pip install nltk
import nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")
# this all are modules

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)

#### Stemming

In [16]:
from nltk.stem import PorterStemmer

In [17]:
def stemming (text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)


#### Encoding the target value

In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [20]:
Y=df["sentiment"]
Y.head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

#### Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [23]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4306699 stored elements and shape (49582, 5000)>

In [24]:
df.head()

,review,sentiment
0,one review nti wtchg 1 oz epod ll hook they ri...,1
1,a wder ltle producti br br t film techniqu uns...,1
2,i thought wder wy spend time o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### TorchDataset and DataLoader

In [25]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [26]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [27]:
x_train=x_train.toarray()
x_test=x_test.toarray()

In [28]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()   # this create a connections
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()  
)

In [29]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=False, batch_size=64)

### RNN 


In [30]:
import torch.nn  as nn
import torch.optim as optim

In [41]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [44]:
input_size=x_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

## Training RNN

In [45]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.473848432302475
epoch = 2/10 and loss = 0.4416015148162842
epoch = 3/10 and loss = 0.1375727355480194
epoch = 4/10 and loss = 0.27559033036231995
epoch = 5/10 and loss = 0.20571815967559814
epoch = 6/10 and loss = 0.3259192407131195
epoch = 7/10 and loss = 0.12706983089447021
epoch = 8/10 and loss = 0.13244293630123138
epoch = 9/10 and loss = 0.24334239959716797
epoch = 10/10 and loss = 0.15850849449634552


### Evaluations

In [49]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 86.0945850559645
